# 🚢 Satellite Boat Detection — YOLOv8m

End-to-end inference pipeline for satellite boat detection using a fine-tuned YOLOv8m model.

**Classes:** `small_boat` · `large_boat` · `ship` · `boat_cluster`

**Model metrics (v3):**
| Metric | Score |
|---|---|
| mAP@50 | 0.8696 |
| mAP@50-95 | 0.5657 |
| Precision | 0.8793 |
| Recall | 0.8265 |

---
**Requirements:** Google Colab (GPU runtime recommended) + Google Drive with model checkpoint.

## 0 — Setup

In [ ]:
# Install dependencies
!pip install ultralytics rasterio geopandas shapely --quiet

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
import os, glob, random, json, shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
from ultralytics import YOLO

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1 — Configuration
**Edit these paths to match your Google Drive layout.**

In [ ]:
# ── Paths ────────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive'
DATASET_PATH  = f'{DRIVE_ROOT}/Ship_dataset'
CHECKPOINT    = None   # set to full path, e.g. f'{DRIVE_ROOT}/boat_best_v3_0425_1234.pt'
               # or leave None to auto-detect latest boat_best_v3_*.pt

# ── Classes ──────────────────────────────────────────
CLASS_NAMES = ['small_boat', 'large_boat', 'ship', 'boat_cluster']
COLORS      = ['#00CC66',    '#FF4444',    '#4488FF', '#FF9900']

# ── Inference settings ───────────────────────────────
CONF_THRESH = 0.35
IOU_THRESH  = 0.45
IMG_SIZE    = 256

print('✅ Config set')

## 2 — Load Model

In [ ]:
# Auto-detect latest checkpoint if not set manually
if CHECKPOINT is None:
    candidates = sorted(glob.glob(f'{DRIVE_ROOT}/boat_best_v3_*.pt'))
    if not candidates:
        raise FileNotFoundError(
            'No boat_best_v3_*.pt found in Drive root. '
            'Set CHECKPOINT manually in the config cell.'
        )
    CHECKPOINT = candidates[-1]

print(f'Loading: {CHECKPOINT}')
model = YOLO(CHECKPOINT)
model.info()
print('✅ Model loaded')

## 3 — Dataset Check

In [ ]:
# Write data.yaml
data_yaml = f"""
path: {DATASET_PATH}
train: images/train
val: images/val

names:
  0: small_boat
  1: large_boat
  2: ship
  3: boat_cluster
"""
with open('data.yaml', 'w') as f:
    f.write(data_yaml)

# Verify splits
from collections import Counter
for split in ['train', 'val']:
    img_dir = f'{DATASET_PATH}/images/{split}'
    lbl_dir = f'{DATASET_PATH}/labels/{split}'
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 'MISSING'
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 'MISSING'
    print(f'{split}: {n_img} images | {n_lbl} labels')

# Class distribution
print('\nClass distribution (train):')
lbl_dir = f'{DATASET_PATH}/labels/train'
counter = Counter()
for f in os.listdir(lbl_dir):
    with open(os.path.join(lbl_dir, f)) as fh:
        for line in fh:
            if line.strip():
                counter[int(line.split()[0])] += 1
for cid, name in enumerate(CLASS_NAMES):
    print(f'  {name:<15} {counter.get(cid, 0):>5} instances')

## 4 — Evaluate on Validation Set

In [ ]:
metrics = model.val(data='data.yaml', conf=0.001, iou=IOU_THRESH, verbose=False)

print('📊 Overall Metrics:')
print(f'  mAP@50:     {metrics.box.map50:.4f}')
print(f'  mAP@50-95:  {metrics.box.map:.4f}')
print(f'  Precision:  {metrics.box.mp:.4f}')
print(f'  Recall:     {metrics.box.mr:.4f}')

print('\n📦 Per-Class AP@50:')
for name, ap in zip(CLASS_NAMES, metrics.box.ap50):
    flag = '⚠️ ' if ap < 0.5 else '✅ '
    print(f'  {flag}{name:<15} {ap:.4f}')

In [ ]:
# Show saved plots (confusion matrix, PR curve, F1 curve)
from IPython.display import Image as IPImage, display

run_dir = sorted(glob.glob('runs/detect/val*'))[-1]
print(f'Results in: {run_dir}')

for plot in ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    path = f'{run_dir}/{plot}'
    if os.path.exists(path):
        print(f'\n{plot}')
        display(IPImage(path, width=600))

## 5 — Ground Truth vs Prediction (Side by Side)

In [ ]:
N_SAMPLES      = 3   # images per frame (2 or 3 recommended)
val_images_dir = f'{DATASET_PATH}/images/val'
val_labels_dir = f'{DATASET_PATH}/labels/val'

# Sample random tiles
all_imgs = [f for f in os.listdir(val_images_dir)
            if f.endswith(('.jpg', '.png', '.tif'))]
samples = random.sample(all_imgs, min(N_SAMPLES, len(all_imgs)))

# Run predictions
pred_dict = {}
for img_file in samples:
    r = model.predict(
        os.path.join(val_images_dir, img_file),
        conf=CONF_THRESH, iou=IOU_THRESH, verbose=False
    )[0]
    pred_dict[img_file] = r.boxes

# Plot
fig, axes = plt.subplots(N_SAMPLES, 2,
                          figsize=(12, N_SAMPLES * 5.5), dpi=120)
if N_SAMPLES == 1:
    axes = [axes]

for row, img_file in enumerate(samples):
    img_path   = os.path.join(val_images_dir, img_file)
    label_path = os.path.join(val_labels_dir,
                              img_file.rsplit('.', 1)[0] + '.txt')
    img = Image.open(img_path).convert('RGB')
    W, H = img.size

    for col, title in enumerate(['Ground truth', 'Prediction']):
        ax = axes[row][col]
        ax.imshow(img, interpolation='nearest')
        ax.set_title(f'{img_file}\n{title}', fontsize=8, pad=4,
                     color='#222222' if col == 0 else '#0055cc')
        ax.axis('off')

        if col == 0 and os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                    x1 = (xc - bw / 2) * W
                    y1 = (yc - bh / 2) * H
                    ax.add_patch(patches.Rectangle(
                        (x1, y1), bw * W, bh * H,
                        linewidth=1.8, edgecolor=COLORS[cls], facecolor='none'
                    ))
                    ax.text(x1 + 2, y1 - 4, CLASS_NAMES[cls],
                            color='white', fontsize=7, fontweight='bold',
                            bbox=dict(boxstyle='square,pad=0.15',
                                      fc=COLORS[cls], ec='none', alpha=0.85))

        if col == 1 and img_file in pred_dict:
            for box in pred_dict[img_file]:
                cls  = int(box.cls[0])
                conf = float(box.conf[0])
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                ax.add_patch(patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=1.8, edgecolor=COLORS[cls],
                    facecolor='none', linestyle='--'
                ))
                ax.text(x1 + 2, y1 - 4,
                        f'{CLASS_NAMES[cls]} {conf:.2f}',
                        color='white', fontsize=7, fontweight='bold',
                        bbox=dict(boxstyle='square,pad=0.15',
                                  fc=COLORS[cls], ec='none', alpha=0.85))

legend_handles = [
    patches.Patch(facecolor=c, label=n)
    for c, n in zip(COLORS, CLASS_NAMES)
] + [
    patches.Patch(facecolor='none', edgecolor='#555', label='solid = GT'),
    patches.Patch(facecolor='none', edgecolor='#555',
                  linestyle='--', label='dashed = pred'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=6,
           fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.suptitle('Ground truth  vs  Prediction', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout(h_pad=2.5)
plt.savefig('gt_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: gt_vs_pred.png')

## 6 — Run Inference on Custom Image or Folder

In [ ]:
# ── Set your input here ──────────────────────────────
# Can be: a single image path, a folder path, or a .tif tile
INFERENCE_SOURCE = f'{DATASET_PATH}/images/val'  # change to your image/folder

results = model.predict(
    source=INFERENCE_SOURCE,
    conf=CONF_THRESH,
    iou=IOU_THRESH,
    imgsz=IMG_SIZE,
    save=True,
    save_txt=True,
    name='boat_inference'
)

total_detections = sum(len(r.boxes) for r in results)
print(f'✅ Processed {len(results)} images')
print(f'   Total detections: {total_detections}')
print(f'   Results saved to: runs/detect/boat_inference/')

In [ ]:
# Show sample predictions
from IPython.display import Image as IPImage, display

pred_imgs = sorted(glob.glob('runs/detect/boat_inference/*.jpg'))[:3]
for p in pred_imgs:
    print(os.path.basename(p))
    display(IPImage(p, width=500))

## 7 — GeoJSON Export (Geo Back-Projection)
Converts pixel bounding boxes → real-world coordinates.
Only works if your source tiles are GeoTIFF (`.tif`) with valid CRS.

In [ ]:
import rasterio
from rasterio.transform import xy

# ── Set your GeoTIFF tiles folder ────────────────────
TILES_DIR    = f'{DRIVE_ROOT}/tiles'   # folder containing .tif tiles
GEOJSON_OUT  = 'boat_detections.geojson'

def pixel_box_to_geo(tile_path, xc_n, yc_n, bw_n, bh_n, img_w, img_h):
    """Convert normalised YOLO box to geo bounding polygon."""
    with rasterio.open(tile_path) as src:
        t   = src.transform
        crs = str(src.crs)
        px  = xc_n * img_w;  py  = yc_n * img_h
        pw  = bw_n * img_w;  ph  = bh_n * img_h
        x1, y1 = px - pw / 2, py - ph / 2
        x2, y2 = px + pw / 2, py + ph / 2
        lon1, lat1 = xy(t, y1, x1)
        lon2, lat2 = xy(t, y2, x2)
    return {
        'type': 'Feature',
        'geometry': {
            'type': 'Polygon',
            'coordinates': [[
                [lon1, lat1], [lon2, lat1],
                [lon2, lat2], [lon1, lat2],
                [lon1, lat1]
            ]]
        },
        'properties': {'crs': crs}
    }

features = []
tif_files = [f for f in os.listdir(TILES_DIR) if f.endswith('.tif')]

for tile_file in tif_files:
    tile_path = os.path.join(TILES_DIR, tile_file)
    results   = model.predict(tile_path, conf=CONF_THRESH,
                               iou=IOU_THRESH, verbose=False)
    for r in results:
        W, H = r.orig_shape[1], r.orig_shape[0]
        for box in r.boxes:
            xc, yc, bw, bh = box.xywhn[0].tolist()
            cls  = int(box.cls[0])
            conf = float(box.conf[0])
            try:
                feat = pixel_box_to_geo(tile_path, xc, yc, bw, bh, W, H)
                feat['properties']['class'] = CLASS_NAMES[cls]
                feat['properties']['conf']  = round(conf, 4)
                feat['properties']['tile']  = tile_file
                features.append(feat)
            except Exception as e:
                print(f'  ⚠️ Skipped box in {tile_file}: {e}')

geojson = {'type': 'FeatureCollection', 'features': features}
with open(GEOJSON_OUT, 'w') as f:
    json.dump(geojson, f, indent=2)

print(f'✅ {len(features)} detections → {GEOJSON_OUT}')
print(f'   Open in QGIS or geojson.io to inspect')

## 8 — Export Model

In [ ]:
# ── ONNX (universal, CPU/GPU) ────────────────────────
model.export(format='onnx', imgsz=IMG_SIZE, simplify=True)
print('✅ Exported to ONNX')

In [ ]:
# ── TensorRT (fastest on Colab T4 GPU) ───────────────
# Uncomment to run — takes ~3 min to compile
# model.export(format='engine', imgsz=IMG_SIZE, half=True)
# print('✅ Exported to TensorRT')

## 9 — Backup Checkpoint to Drive

In [ ]:
# Run this at the end of any training session
def backup_checkpoint(src_path, drive_root=DRIVE_ROOT):
    stamp = datetime.now().strftime('%m%d_%H%M')
    dst   = f'{drive_root}/boat_best_v3_{stamp}.pt'
    shutil.copy(src_path, dst)
    print(f'✅ Backed up → {dst}')
    return dst

# Example — update path to your latest run:
# backup_checkpoint('/content/runs/detect/boat_v3/weights/best.pt')

---
## Training Reference
To retrain from this checkpoint:

```python
model.train(
    data='data.yaml',
    epochs=100,
    imgsz=256,
    batch=16,
    name='boat_v4',
    optimizer='AdamW',
    lr0=0.0003,
    lrf=0.01,
    warmup_epochs=3,
    freeze=3,
    mosaic=0.8,
    degrees=15.0,
    flipud=0.5,
    fliplr=0.5,
    scale=0.3,
    hsv_h=0.0,
    hsv_s=0.2,
    hsv_v=0.2,
    box=10.0,
    cls=0.3,
    patience=20,
)
```